In [ ]:
# !pip uninstall -y flax orbax-checkpoint jax jaxlib \
#     ml-dtypes tf-keras tensorflow tensorflow-cpu tensorflow-text \
#     tensorflow-decision-forests keras keras-hub \
#     chex optax fastai spacy tensorstore numba \
#     umap-learn pynndescent librosa shap cuml-cu12 cudf-cu12 dask-cuda

%pip install -U threadpoolctl joblib
#%pip install --force-reinstall numpy==1.26.4
%pip install --upgrade scikit-learn
%pip install -U imbalanced-learn umap-learn

%pip install --upgrade --force-reinstall \
    numpy \
    scipy \
    matplotlib \
    seaborn \
    pandas \
    tensorflow

# Загружаем датасет

In [2]:
import pandas as pd
df = pd.read_csv('../dataset/lstm/lstm_sequence.csv')

# 4. Отладочная информация
print("\n===== SHAPE =====")
print(df.shape)

print("\n===== HEAD (5 строк) =====")
display(df.head())

print("\n===== Классовое соотношение (isFraud) =====")
if "FLAG" in df.columns:
    display((df["FLAG"].value_counts(normalize=True) * 100).round(2).rename("%"))
else:
    print("Столбец 'FLAG' не найден — проверьте структуру датасета.")


===== SHAPE =====
(1106250, 42)

===== HEAD (5 строк) =====


,address,FLAG,FLAG_NUM,is_contract,scam_type,step_idx,window_start,Sent_tnx,Received_tnx,Number_of_Created_Contracts,...,ERC20_Total_Ether_Sent_Contract,ERC20_Uniq_Sent_Addr,ERC20_Uniq_Rec_Addr,ERC20_Uniq_Rec_Contract_Addr,ERC20_Min_Val_Rec,ERC20_Max_Val_Rec,ERC20_Avg_Val_Rec,ERC20_Min_Val_Sent,ERC20_Max_Val_Sent,ERC20_Avg_Val_Sent
0,0x5581c4791b29561985ce4c676e3d72a09b59d267,scam,1.0,False,NaN,28,2024-12-19,0,0,0,...,0.0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
1,0x5581c4791b29561985ce4c676e3d72a09b59d267,scam,1.0,False,NaN,29,2024-12-26,0,0,0,...,0.0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,0x5584d5ab83d9e338a89769c66df095212bc528e3,scam,1.0,False,NaN,0,2024-06-06,0,0,0,...,0.0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,0x5584d5ab83d9e338a89769c66df095212bc528e3,scam,1.0,False,NaN,1,2024-06-13,0,0,0,...,0.0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
4,0x5584d5ab83d9e338a89769c66df095212bc528e3,scam,1.0,False,NaN,2,2024-06-20,0,0,0,...,0.0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0



===== Классовое соотношение (isFraud) =====


FLAG
legit      73.58
scam       20.52
false       5.42
suspect     0.49
Name: %, dtype: float64

In [7]:
import pandas as pd, numpy as np

RAW_PATH = "../dataset/lstm/lstm_sequence.csv"      # ваш загруженный CSV

df = (
    pd.read_csv(RAW_PATH,
                parse_dates=['window_start']) # пригодится, если захотите
      .rename(columns=str.strip)
)

# 1) берём только смарт-контракты
df = df[df['is_contract'].astype(bool)]

# 2) целевая переменная
df['FLAG'] = df['FLAG'].map({'scam': 1, 'legit': 0}).astype(np.int8)

print(df.head)

IntCastingNaNError: Cannot convert non-finite values (NA or inf) to integer

In [ ]:
import pandas as pd

# Загрузка данных
df = pd.read_csv("../dataset/lstm/lstm_sequence.csv")

# Маска для FLAG == 'false'
mask_false = df['FLAG'] == 'false'
df.loc[mask_false, 'FLAG'] = 'legit'
df.loc[mask_false, 'FLAG_NUM'] = 0

# Маска для FLAG == 'true'
mask_true = df['FLAG'] == 'true'
df.loc[mask_true, 'FLAG'] = 'scam'
df.loc[mask_true, 'FLAG_NUM'] = 1

# Явное приведение FLAG_NUM к целому типу
df['FLAG_NUM'] = df['FLAG_NUM'].astype(int)

# Сохранение обратно в файл с перезаписью
df.to_csv("../dataset/lstm/lstm_sequence.csv", index=False)
